In [95]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

# Preparation of Data Frames

In [96]:
df  = pd.DataFrame()

## Sentiment

In [97]:
# Reading the saved data pickle file

df_sentiment  = pd.DataFrame()

df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticles.pkl', compression=None)
# df_sentiment.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
df_sentiment.drop(columns=["Company"], inplace=True)

df_sentiment.index = pd.to_datetime(df_sentiment.index)
df_sentiment.Date = pd.to_datetime(df_sentiment.Date)

df_sentiment = df_sentiment.ffill()

df_sentiment["sentiment(t-1)"] = df_sentiment["sentiment"].shift(periods=1)
df_sentiment["sentiment(t-2)"] = df_sentiment["sentiment"].shift(periods=2)
df_sentiment["sentiment(t-1)"] = df_sentiment["sentiment(t-1)"].ffill()
df_sentiment["sentiment(t-2)"] = df_sentiment["sentiment(t-2)"].ffill()

# SENTIMENT RETURN
df_sentiment["sentiment_return"]=(df_sentiment["sentiment(t-1)"]-df_sentiment["sentiment(t-2)"])/df_sentiment["sentiment(t-2)"]
# SENTIMENT LN RETURN
df_sentiment["sentiment_ln_return"]=np.log(df_sentiment["sentiment(t-1)"]/df_sentiment["sentiment(t-2)"])
# SENTIMENT EWM
df_sentiment["sentiment_ewm"]=df_sentiment["sentiment"].ewm(10).mean()

In [98]:
df_sentiment = df_sentiment[["Date", "sentiment", "sentiment(t-1)", "sentiment(t-2)", "sentiment_return", "sentiment_ln_return", "sentiment_ewm", "PressReleaseCount"]]

In [99]:
df = df_sentiment

In [100]:
df

,Date,sentiment,sentiment(t-1),sentiment(t-2),sentiment_return,sentiment_ln_return,sentiment_ewm,PressReleaseCount
2019-09-25,2019-09-25,94.117647,NaN,NaN,NaN,NaN,94.117647,13
2019-09-26,2019-09-26,85.826772,94.117647,NaN,NaN,NaN,89.774808,11
2019-09-27,2019-09-27,94.545455,85.826772,94.117647,-0.088091,-0.092215,91.518760,7
2019-09-30,2019-09-30,90.000000,94.545455,85.826772,0.101585,0.096750,91.083192,10
2019-10-01,2019-10-01,100.000000,90.000000,94.545455,-0.048077,-0.049271,93.221585,12
...,...,...,...,...,...,...,...,...
2021-08-19,2021-08-19,71.779141,68.047337,81.595092,-0.166036,-0.181566,77.598536,25
2021-08-20,2021-08-20,74.545455,71.779141,68.047337,0.054841,0.053390,77.320983,13
2021-08-23,2021-08-23,74.698795,74.545455,71.779141,0.038539,0.037815,77.082602,29
2021-08-24,2021-08-24,85.326087,74.698795,74.545455,0.002057,0.002055,77.832010,30


## VIX

In [101]:
from yahoo_fin.stock_info import get_data

df_vix  = pd.DataFrame()
df_vix = get_data("^VIX", start_date="25/09/2019", end_date="26/08/2021", index_as_date = True, interval="1d")
df_vix.index = pd.to_datetime(df_vix.index)
df_vix["Date"]= pd.to_datetime(df_vix.index)

df_vix["vix"] = df_vix["adjclose"]
df_vix.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_vix["vix"]  = df_vix["vix"].ffill()

df_vix["vix_yesterday"] = df_vix["vix"].shift(periods=1)
# vix RETURN
df_vix["vix_return"]=(df_vix["vix"]-df_vix["vix_yesterday"])/df_vix["vix_yesterday"]
# vix LN RETURN
df_vix["vix_ln_return"]=np.log(df_vix["vix"]/df_vix["vix_yesterday"])
# vix EWM
df_vix["vix_ewm"]=df_vix["vix"].ewm(10).mean()

In [102]:
df = pd.merge(df_vix,df,how='outer', on='Date')

## S&P500

In [103]:
df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date="26/08/2021", index_as_date = True, interval="1d")

In [104]:
df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date="26/08/2021", index_as_date = True, interval="1d")
df_sp500.index = pd.to_datetime(df_sp500.index)
df_sp500["Date"]= pd.to_datetime(df_sp500.index)

df_sp500["sp500"] = df_sp500["adjclose"]
df_sp500.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_sp500["sp500"]  = df_sp500["sp500"].ffill()

df_sp500["sp500_yesterday"] = df_sp500["sp500"].shift(periods=1)
# sp500 RETURN
df_sp500["sp500_return"]=(df_sp500["sp500"]-df_sp500["sp500_yesterday"])/df_sp500["sp500_yesterday"]
# sp500 LN RETURN
df_sp500["sp500_ln_return"]=np.log(df_sp500["sp500"]/df_sp500["sp500_yesterday"])
# sp500 EWM
df_sp500["sp500_ewm"]=df_sp500["sp500"].ewm(10).mean()

In [105]:
df = pd.merge(df_sp500,df,how='outer', on='Date')

In [106]:
# df = df.drop([475])

# DAY MONTH YEAR

In [107]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['year'] = [d.year for d in df.Date]
df['month'] = [d.strftime('%b') for d in df.Date]
df['day'] = [d.day_name() for d in df.Date]
years = df['year'].unique()
df['month-year'] = [d.strftime('%b') + str(d.year) for d in df.Date]

# THREE PERIODS

In [108]:
df = df.ffill()
df.index = df.Date

In [109]:
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':'2021-08-25']

In [110]:
df_part3["sentiment(t-1)"] = df_part3["sentiment"].shift(periods=1)
df_part3["sentiment(t-2)"] = df_part3["sentiment"].shift(periods=2)


# SENTIMENT RETURN
df_part3["sentiment_return"]=(df_part3["sentiment(t-1)"]-df_part3["sentiment(t-2)"])/df_part3["sentiment(t-2)"]
# SENTIMENT LN RETURN
df_part3["sentiment_ln_return"]=np.log(df_part3["sentiment(t-1)"]/df_part3["sentiment(t-2)"])
# SENTIMENT EWM
df_part3["sentiment_ewm"]=df_part3["sentiment"].ewm(10).mean()

# TEXTBLOB

In [111]:
# Reading the saved data pickle file

del df_sentiment
df_sentiment  = pd.DataFrame()

df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticlesTextBlob.pkl', compression=None)
# df_sentiment.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
df_sentiment.drop(columns=["Company", "PressReleaseCount", "Articles", "Title", "Text"], inplace=True)

df_sentiment.index = pd.to_datetime(df_sentiment.index)
df_sentiment.Date = pd.to_datetime(df_sentiment.Date)

df_sentiment = df_sentiment.ffill()

df_sentiment["subjectivity(t-1)"] = df_sentiment["subjectivity"].shift(periods=1)
df_sentiment["subjectivity(t-2)"] = df_sentiment["subjectivity"].shift(periods=2)

# SENTIMENT RETURN
df_sentiment["subjectivity_return"]=(df_sentiment["subjectivity(t-1)"]-df_sentiment["subjectivity(t-2)"])/df_sentiment["subjectivity(t-2)"]
# SENTIMENT LN RETURN
df_sentiment["subjectivity_ln_return"]=np.log(df_sentiment["subjectivity(t-1)"]/df_sentiment["subjectivity(t-2)"])
# SENTIMENT EWM
df_sentiment["subjectivity_ewm"]=df_sentiment["subjectivity"].ewm(10).mean()

In [113]:
df.reset_index(drop=True, inplace=True)
df_sentiment.reset_index(drop=True, inplace=True)
df_sentiment.ffill()

,polarity,subjectivity,PosNegNeu,Date,subjectivity(t-1),subjectivity(t-2),subjectivity_return,subjectivity_ln_return,subjectivity_ewm
0,0.022975,0.254230,Positive,2019-09-25,NaN,NaN,NaN,NaN,0.254230
1,0.017622,0.266415,Positive,2019-09-26,0.254230,NaN,NaN,NaN,0.260612
2,0.047938,0.412416,Positive,2019-09-27,0.266415,0.254230,0.047930,0.046816,0.316106
3,0.031246,0.367310,Positive,2019-09-30,0.412416,0.266415,0.548021,0.436977,0.330791
4,0.105016,0.415282,Positive,2019-10-01,0.367310,0.412416,-0.109370,-0.115826,0.351053
...,...,...,...,...,...,...,...,...,...
458,0.070968,0.313283,Positive,2021-08-19,0.349444,0.335961,0.040134,0.039350,0.344665
459,0.009536,0.306991,Positive,2021-08-20,0.313283,0.349444,-0.103482,-0.109237,0.341240
460,0.061468,0.339064,Positive,2021-08-23,0.306991,0.313283,-0.020084,-0.020289,0.341042
461,0.131828,0.339332,Positive,2021-08-24,0.339064,0.306991,0.104475,0.099370,0.340886


In [115]:
df

,Date,sp500,sp500_yesterday,sp500_return,sp500_ln_return,sp500_ewm,vix,vix_yesterday,vix_return,vix_ln_return,...,sentiment(t-1),sentiment(t-2),sentiment_return,sentiment_ln_return,sentiment_ewm,PressReleaseCount,year,month,day,month-year
0,2019-09-25,2984.870117,NaN,NaN,NaN,2984.870117,15.960000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,94.117647,13.0,2019,Sep,Wednesday,Sep2019
1,2019-09-26,2977.620117,2984.870117,-0.002429,-0.002432,2981.072498,16.070000,15.960000,0.006892,0.006869,...,94.117647,NaN,NaN,NaN,89.774808,11.0,2019,Sep,Thursday,Sep2019
2,2019-09-27,2961.790039,2977.620117,-0.005316,-0.005331,2974.023623,17.219999,16.070000,0.071562,0.069117,...,85.826772,94.117647,-0.088091,-0.092215,91.518760,7.0,2019,Sep,Friday,Sep2019
3,2019-09-30,2976.739990,2961.790039,0.005048,0.005035,2974.802655,16.240000,17.219999,-0.056911,-0.058594,...,94.545455,85.826772,0.101585,0.096750,91.083192,10.0,2019,Sep,Monday,Sep2019
4,2019-10-01,2940.250000,2976.739990,-0.012258,-0.012334,2966.516379,18.559999,16.240000,0.142857,0.133531,...,90.000000,94.545455,-0.048077,-0.049271,93.221585,12.0,2019,Oct,Tuesday,Oct2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479,2021-08-19,4405.799805,4400.270020,0.001257,0.001256,4413.914368,21.670000,21.570000,0.004636,0.004625,...,68.047337,81.595092,-0.166036,-0.181566,77.598536,25.0,2021,Aug,Thursday,Aug2021
480,2021-08-20,4441.669922,4405.799805,0.008142,0.008109,4416.437600,18.559999,21.670000,-0.143516,-0.154920,...,71.779141,68.047337,0.054841,0.053390,77.320983,13.0,2021,Aug,Friday,Aug2021
481,2021-08-23,4479.529785,4441.669922,0.008524,0.008488,4422.173254,17.150000,18.559999,-0.075970,-0.079011,...,74.545455,71.779141,0.038539,0.037815,77.082602,29.0,2021,Aug,Monday,Aug2021
482,2021-08-24,4486.229980,4479.529785,0.001496,0.001495,4427.996592,17.219999,17.150000,0.004082,0.004073,...,74.698795,74.545455,0.002057,0.002055,77.832010,30.0,2021,Aug,Tuesday,Aug2021


In [116]:
df = pd.merge(df_sentiment,df,how='outer', on='Date')

In [ ]:
df.index = df.Date

In [ ]:
df

# DF HOURLY

In [ ]:
df_time=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticlesTime.pkl', compression=None)
df_hour=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticlesDateHour.pkl', compression=None)

In [ ]:
df_hour.Date= pd.to_datetime(df_hour.Date)

In [ ]:
df_hour = df_hour.merge(df_sp500, on='Date', how='left')

In [ ]:
# Prepare data
df_hour['year'] = [d.year for d in df_hour.Date]
df_hour['month'] = [d.strftime('%b') for d in df_hour.Date]
df_hour['day'] = [d.day_name() for d in df_hour.Date]
years = df_hour['year'].unique()
df_hour['month-year'] = [d.strftime('%b') + str(d.year) for d in df_hour.Date]

In [ ]:
df_hour.index = df_hour.Date

In [ ]:
df_hour.sort_values(by="Hour")

In [ ]:
df_hour_part1 = df_hour.loc['2019-09-25':'2020-02-14']
df_hour_part2 = df_hour.loc['2020-02-15':'2020-05-31']
df_hour_part3 = df_hour.loc['2020-06-01':'2021-08-25']

In [ ]:
df_hour["sentiment"]

# 09-16

In [ ]:
df_09_15=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticles_09_15.pkl', compression=None)
df_16_08=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticles_16_08.pkl', compression=None)
df_09_15.index= pd.to_datetime(df_09_15.index)
df_16_08.index= pd.to_datetime(df_16_08.index)
df_09_15.Date= pd.to_datetime(df_09_15.Date)
df_16_08.Date= pd.to_datetime(df_16_08.Date)

In [ ]:
# Prepare data df_09_15
df_09_15['year'] = [d.year for d in df_09_15.Date]
df_09_15['month'] = [d.strftime('%b') for d in df_09_15.Date]
df_09_15['day'] = [d.day_name() for d in df_09_15.Date]
years = df_09_15['year'].unique()
df_09_15['month-year'] = [d.strftime('%b') + str(d.year) for d in df_09_15.Date]

# Prepare data df_16_08
df_16_08['year'] = [d.year for d in df_16_08.Date]
df_16_08['month'] = [d.strftime('%b') for d in df_16_08.Date]
df_16_08['day'] = [d.day_name() for d in df_16_08.Date]
years = df_16_08['year'].unique()
df_16_08['month-year'] = [d.strftime('%b') + str(d.year) for d in df_16_08.Date]


In [ ]:
df_09_15 = df_09_15.merge(df_vix,on='Date',how='left')
df_09_15 = df_09_15.merge(df_sp500, on='Date', how='left')

df_16_08 = df_16_08.merge(df_vix,on='Date',how='left')
df_16_08 = df_16_08.merge(df_sp500, on='Date', how='left')

In [ ]:
df_09_15.index= pd.to_datetime(df_09_15.Date)
df_16_08.index= pd.to_datetime(df_16_08.Date)
df_09_15.Date= pd.to_datetime(df_09_15.Date)
df_16_08.Date= pd.to_datetime(df_16_08.Date)

In [ ]:
df_09_15["sentiment"] = df_09_15["sentiment"].apply(np.float)
df_16_08["sentiment"] = df_16_08["sentiment"].apply(np.float)

In [ ]:
sentiment_yesterday = df_09_15["sentiment"].shift(periods=1)
sentiment_yesterday = sentiment_yesterday.ffill()
df_09_15["sentiment_ln_return"]=np.log(df_09_15["sentiment"]/sentiment_yesterday)


sentiment_yesterday = df_16_08["sentiment"].shift(periods=1)
sentiment_yesterday = sentiment_yesterday.ffill()
df_16_08["sentiment_ln_return"]=np.log(df_16_08["sentiment"]/sentiment_yesterday)

In [ ]:
df_09_15["sentiment"] = df_09_15["sentiment"].ffill()
df_16_08["sentiment"] = df_16_08["sentiment"].ffill()

In [ ]:
sentiment_yesterday = df_09_15["sentiment"].shift(periods=1)
df_09_15["sentiment_return"]=((df_09_15["sentiment"]-sentiment_yesterday)/sentiment_yesterday)
sentiment_yesterday = df_16_08["sentiment"].shift(periods=1)
df_16_08["sentiment_return"]=((df_16_08["sentiment"]-sentiment_yesterday)/sentiment_yesterday)

In [ ]:
df_09_15["sentiment_return"] = df_09_15["sentiment_return"].ffill()
df_16_08["sentiment_return"] = df_16_08["sentiment_return"].ffill()

In [ ]:
# df_part3['Num_word'] = df_part3['Articles'].apply(lambda x:len(str(x).split())) #Number Of words in main text